# Sentence embedding

- fixed length numerical representence of a document which carry semantic meaning and structure


## 1. Averging Sentence Embedding

- take average of all the word embedding

**Advantages:**
    - Simple and fast calculation

**Disadvantages:**
    - It treats all the words equally , same weights to all the words

In [ ]:
import spacy
import numpy as np

# Load a pre-trained word embedding model (spaCy's small English model)
nlp = spacy.load("en_core_web_sm")


In [ ]:
def average_embedding(sentence):
    doc = nlp(sentence)
    vectors = []
    for token in doc:
        if token.has_vector:  # Exclude stopwords without vectors
            vectors.append(token.vector)

    # return average vector
    if vectors:
        return np.mean(vectors, axis=0)

    # fallback if no vectors found
    return np.zeros((nlp.vocab.vectors_length,))

# Example sentence
sentence = "The dog runs fast"
embedding = average_embedding(sentence)

print("Sentence Embedding (Averaging):", embedding[:5])  # Print first 5 values

Sentence Embedding (Averaging): [ 0.16034189 -0.06278078 -0.51387084 -0.10410061  0.00280234]


#### Task using this
- sentence retrieval

In [1]:
# dataset
sentences = [
    "The dog is running in the park",
    "A cat is sleeping on the sofa",
    "Children are playing football outside",
    "The car is moving very fast",
    "A man is cooking dinner in the kitchen",
    "Birds are flying in the sky",
    "The puppy runs quickly",
    "Students are studying in the library",
    "The train arrived at the station",
    "A woman is reading a book"
]

input_sentence = "The dog runs fast"

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

input_embedding = average_embedding(input_sentence)

# compare this with all the sentences of dataset
best_sentence = None
best_score = -1

print(f"Input Sentence: {input_sentence}")

for sent in sentences:
    sent_embedding = average_embedding(sent)

    # cosine similarity

    score = cosine_similarity(
        [input_embedding],
        [sent_embedding]
    )[0][0]

    print(f"Sentence: {sent}")
    print(f"Similarity score: {score:.4f}")

    if score > best_score:
        best_score = score
        best_sentence = sent
print("Most similar Sentence:")
print(best_sentence)

print("Best Similarity Score:")
print(round(best_score, 4))

Input Sentence: The dog runs fast
Sentence: The dog is running in the park
Similarity score: 0.4512
Sentence: A cat is sleeping on the sofa
Similarity score: 0.4390
Sentence: Children are playing football outside
Similarity score: 0.2932
Sentence: The car is moving very fast
Similarity score: 0.5671
Sentence: A man is cooking dinner in the kitchen
Similarity score: 0.4061
Sentence: Birds are flying in the sky
Similarity score: 0.3213
Sentence: The puppy runs quickly
Similarity score: 0.8413
Sentence: Students are studying in the library
Similarity score: 0.2535
Sentence: The train arrived at the station
Similarity score: 0.5311
Sentence: A woman is reading a book
Similarity score: 0.4654
Most similar Sentence:
The puppy runs quickly
Best Similarity Score:
0.8413


In [ ]:
# cosine_similarity(
#     [average_embedding("hello world")],
#     [average_embedding("hello! world")]
# )

array([[0.62074435]], dtype=float32)

In [2]:
# creating custom function
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def find_most_similar(input_sentence, sentences, embedding_function, top_k = 1):
    input_embedding = embedding_function(input_sentence)

    results = []

    for sent in sentences:
        sent_embedding = embedding_function(sent)

        score = cosine_similarity(
            [input_embedding],
            [sent_embedding]
        )[0][0]

        results.append((sent, score))

    results.sort(key = lambda x : x[1], reverse = True) # sort by similarity score in descending order
    return results[: top_k]

In [ ]:
print(find_most_similar(input_sentence, sentences, average_embedding))

[('The puppy runs quickly', np.float32(0.841335))]


## 2. Tf-idf Weighted sentence embedding

- more weight to less freqent word in corpus and more frequent word in document
- sentence embedding = summation((tfidf of word * word embedding)/(summation of all the tfidf scores))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()

corpus = sentences
tfidf.fit(corpus)

words = tfidf.get_feature_names_out()

def tfidf_weighted_embedding(sentence):

    # moved inside function
    tfidf_scores = tfidf.transform([sentence]).toarray()[0]

    doc = nlp(sentence)

    word_embedding = []
    weights = []

    for token in doc:

        word = token.text.lower()

        if word in words and token.has_vector:

            idx = list(words).index(word)

            # added weight variable
            weight = tfidf_scores[idx]

            # avoid zero weights
            if weight > 0:

                word_embedding.append(
                    token.vector * weight
                )

                weights.append(weight)

    # safe division
    return (
        np.sum(word_embedding, axis=0) / np.sum(weights)
        if len(weights) > 0
        else np.zeros(nlp.vocab.vectors_length,)
    )

embedding = tfidf_weighted_embedding(sentence)

print("sentence embedding (tf-idf-weighted): ", embedding[:5])

sentence embedding (tf-idf-weighted):  [ 0.12669342 -0.00588569 -0.57719909 -0.23325514  0.08214078]


In [ ]:
top_results = find_most_similar(
    input_sentence,
    sentences,
    tfidf_weighted_embedding,
    top_k=1
)
best_sentence, best_score = top_results[0]

print("Input Sentence:")
print(input_sentence)

print("\nMost Similar Sentence:")
print(best_sentence)

print("\nBest Similarity Score:")
print(round(best_score, 4))

Input Sentence:
The dog runs fast

Most Similar Sentence:
The puppy runs quickly

Best Similarity Score:
0.8128


**Advantages:**
    - Assign higher weight to important words.
    - Reducing noise from the data

**Disadvantages:**
    - Failed to capture deep contextual relationship between words

## 3. PCA & SVD

- find Principal components which captures maximum variance in the data.
- steps to use this:
    1.  Sentence embedding averaging or tfidf weighted embedding
    2. PCA/SVD to remove first principal component
    3. find sentence embedding


In [ ]:
from sklearn.decomposition import PCA

# Generate embeddings for all sentences in corpus
sentence_embeddings = np.array([average_embedding(sent) for sent in sentences])

# apply PCA to reduce dimensions (e.g. from 300 to 50)
pca = PCA()
reduced_embedding = pca.fit_transform(sentence_embeddings)

print("Reduced Sentence Embedding (PCA): ", reduced_embedding[0][:5])

Reduced Sentence Embedding (PCA):  [-0.48826987 -0.64435625  0.05232776  0.25138375  0.05768984]


**Advantages:**
    - It reduces noise and dimensionality

**Disadvantages:**
    - lose context (sentence meaning)

## 4. Contextual Embedding

- till this point all above methods just ignored order of words
- word embedding was independent of other words
- **BUT**
- here transformers use self - attention mechanism for contextual embedding

- 1. I left the room
- 2. left of the room
- here if i apply abover methods then they might consider same left semantic meaning in both sentences which is wrong
- but transformers apply self - attention mechanism and produces contextual meaning, which is correct in both the sentences left is used in different context.

In [ ]:
# !pip install sentence-transformers

In [6]:
from sentence_transformers import SentenceTransformer

# load pre-trained SBERT model
model = SentenceTransformer("all-MiniLM-L6-v2")

# compute SBERT sentence embedding
sentence = "The dog runs fast"
embedding = model.encode(sentence)

print("Sentence Embedding (SBERT): ", embedding[:5])

def sbert_embedding(sentence):
    return model.encode(sentence)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Sentence Embedding (SBERT):  [0.03234608 0.03101395 0.0102431  0.03830507 0.00155678]


In [7]:
top_results = find_most_similar(
    input_sentence,
    sentences,
    sbert_embedding,
    top_k=3
)

print("Input Sentence:")
print(input_sentence)

print("\nTop Similar Sentences:\n")

for sentence, score in top_results:

    print(f"Sentence: {sentence}")
    print(f"Similarity Score: {score:.4f}\n")

Input Sentence:
The dog runs fast

Top Similar Sentences:

Sentence: The puppy runs quickly
Similarity Score: 0.8547

Sentence: The dog is running in the park
Similarity Score: 0.5759

Sentence: The car is moving very fast
Similarity Score: 0.4409

